# 72 — M7 certified scheme search

目的: 許可された scheme 空間から、独立検証付きで \(q_{cert}^{upper}<1\) を満たす scheme を見つける。

M0–M6 の完了 run は不変。`q_cert>=1` は certificate failure であり真の RG 写像の非収縮証明ではない。

- **Campaign A**: majorant / Perron / product（親行列の再重み）。
- **Campaign B**: S2 有限近似（`target_rank` / oversampling / power_iterations）と M3→M6 lineage plan。
- **Campaign C**: S3 geometry/cutoff（`j2_max` / channel_policy / block_geometry）と M2→M6 lineage plan（人間レビュー必須）。


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


In [ ]:
from pathlib import Path
import os, sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
    Path('/storage') / BUNDLE_NAME,
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
    and (p.expanduser() / 'src').is_dir()
    and (p.expanduser() / 'tests').is_dir()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('M7 project root was not found.')
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_M6_RUN_ID', 'M6-20260720T061700Z-7c4e91a2b850')
os.environ.setdefault('VALIDATED_RG_M7_RUN_ID', 'M7-20260720T074400Z-9f2a18c6d401')
os.environ.setdefault('VALIDATED_RG_M7B_RUN_ID', 'M7-20260720T080000Z-b7c4e91a2b85')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
import os
from pathlib import Path
import pytest

os.chdir(PROJECT_ROOT)
test_files = [
    PROJECT_ROOT / 'tests/test_m7_orchestrator.py',
    PROJECT_ROOT / 'tests/test_m6_orchestrator.py',
]
missing = [str(p) for p in test_files if not p.is_file()]
if missing:
    raise RuntimeError(f'Missing tests: {missing}')
rc = pytest.main(['-q', f'--rootdir={PROJECT_ROOT}', *[str(p) for p in test_files]])
if rc != 0:
    raise RuntimeError(f'M7 CPU suite failed: {rc}')
print('M7 CPU suite: PASS')


In [ ]:
from src.m7_config import default_m7_config
from src.m7_orchestrator import create_or_resume_m7
from src.m7_status import M6_PARENT_RUN_ID_FROZEN, M7_RUN_ID_FROZEN
import os
from pathlib import Path

project_root = Path(os.environ['VALIDATED_RG_PROJECT_ROOT'])
persist_root = Path(os.environ['VALIDATED_RG_PERSIST_ROOT'])
config = default_m7_config(
    parent_m6_run_id=os.environ.get('VALIDATED_RG_M6_RUN_ID', M6_PARENT_RUN_ID_FROZEN),
    run_id=os.environ.get('VALIDATED_RG_M7_RUN_ID', M7_RUN_ID_FROZEN),
    mode='paperspace',
    campaign='A',
    max_candidates_total=32,
    max_rigorous_replays=16,
    stop_on_first_certified=True,
)
m7 = create_or_resume_m7(persist_root, config, project_root)
M7_RESULT = m7.run_search()
M7_RESULT


In [ ]:
# Campaign B — S2 rank/residual lineage plans (after Campaign A exhausted)
from src.m7_config import default_m7_config
from src.m7_orchestrator import create_or_resume_m7
from src.m7_status import M6_PARENT_RUN_ID_FROZEN, M7_RUN_ID_CAMPAIGN_B
import os
from pathlib import Path

project_root = Path(os.environ['VALIDATED_RG_PROJECT_ROOT'])
persist_root = Path(os.environ['VALIDATED_RG_PERSIST_ROOT'])
config_b = default_m7_config(
    parent_m6_run_id=os.environ.get('VALIDATED_RG_M6_RUN_ID', M6_PARENT_RUN_ID_FROZEN),
    run_id=os.environ.get('VALIDATED_RG_M7B_RUN_ID', M7_RUN_ID_CAMPAIGN_B),
    mode='paperspace',
    campaign='B',
    lineage_mode='plan_only',  # emit LOCK + rigorous_lineage.json; no fake CERTIFIED
    max_candidates_total=32,
    max_rigorous_replays=16,
    max_lineage_replays=2,
    stop_on_first_certified=True,
)
m7b = create_or_resume_m7(persist_root, config_b, project_root)
M7B_RESULT = m7b.run_search()
M7B_RESULT


In [ ]:
# Campaign C — S3 geometry/cutoff lineage plans (after B core-dominated)
from src.m7_config import default_m7_config
from src.m7_orchestrator import create_or_resume_m7
from src.m7_status import M6_PARENT_RUN_ID_FROZEN, M7_RUN_ID_CAMPAIGN_C
import os
from pathlib import Path

project_root = Path(os.environ['VALIDATED_RG_PROJECT_ROOT'])
persist_root = Path(os.environ['VALIDATED_RG_PERSIST_ROOT'])
config_c = default_m7_config(
    parent_m6_run_id=os.environ.get('VALIDATED_RG_M6_RUN_ID', M6_PARENT_RUN_ID_FROZEN),
    run_id=os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C),
    mode='paperspace',
    campaign='C',
    lineage_mode='plan_only',
    human_review_approved=False,  # set True only after explicit human review
    max_candidates_total=32,
    max_rigorous_replays=16,
    max_lineage_replays=2,
    stop_on_first_certified=True,
)
m7c = create_or_resume_m7(persist_root, config_c, project_root)
M7C_RESULT = m7c.run_search()
M7C_RESULT


## 解釈

- `CERTIFIED_SCHEME_FOUND`: 独立 verifier が `q_cert_upper < 1` を再確認。成功。
- `M7_SEARCH_SPACE_EXHAUSTED`: ロック済み探索空間では証明書なし。理論的不存在は主張しない。
- `M7_LINEAGE_PLANNED`: Campaign B/C が lineage plan を出力。`plan_only` では CERTIFIED を出さない。
- Campaign C は人間レビュー後に execute。`j2_max>1` は現行 M2/M3 が fail-closed のため、実行前に unlock が必要。
- 親と同じ `q` の継承は D0。S2 だけでは core 支配なら閉じない。S3 は代数/geometry を変える。
